# 01. Carregamento e Limpeza de Dados

**Input:** `data/telemetry_*.parquet`, `data/desenvolver_apontamentos.parquet`  
**Output:** `data/cleaned_telemetry.parquet`, `data/cleaned_apontamentos.parquet`, `data/quality_report.md`

Este notebook documenta cada erro de qualidade encontrado: coluna afetada, contagem de registros, correção aplicada.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from src.utils import set_seeds
from src.loader import load_telemetry
from src.cleaner import clean_telemetry
from src.validator import validate_apontamentos

set_seeds(42)

DATA_DIR = '../data'

## 1. Carregamento da Telemetria

In [ ]:
df_raw = load_telemetry(DATA_DIR)
print(f'Registros carregados: {len(df_raw):,}')
print(f'Meses: {sorted(df_raw["_source_month"].unique())}')
df_raw.head()

In [ ]:
print('Schema:')
print(df_raw.dtypes)
print(f'\nMemória: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')

## 2. Diagnóstico de Erros (pré-limpeza)

In [ ]:
# Erro 1. Criticidade encoding
crit_bugs = df_raw['Criticidade'].str.contains(r'N.{1,2}o Cr.{1,2}tico', regex=True, na=False)
print(f'Erro 1. Criticidade encoding corrompido: {crit_bugs.sum():,} registros')
print(df_raw.loc[crit_bugs, 'Criticidade'].value_counts())

In [ ]:
# Erro 2. NULL string em Classe
null_mask = df_raw['Classe'] == 'NULL'
print(f'Erro 2. String literal NULL em Classe: {null_mask.sum():,} registros')
print(f'  Percentual do mês Jan: {null_mask[df_raw["_source_month"]=="jan"].sum():,}')

In [ ]:
# Erro 3. Separador decimal em Valor
comma_mask = df_raw['Valor'].astype(str).str.contains(',', na=False)
print(f'Erro 3. Vírgula como separador decimal em Valor: {comma_mask.sum():,} registros')
print(df_raw.loc[comma_mask, 'Valor'].head())

## 3. Aplicação da Limpeza

In [ ]:
df_clean, report = clean_telemetry(df_raw)
print(report.summary)

In [ ]:
# Verificação pós-limpeza
# Nota: checamos o marcador de corrupção ('?') em vez do regex usado na correção.
# Esse regex usa '.' (wildcard), que também casa com 'Não Crítico' já corrigido.
assert df_clean['Criticidade'].astype(str).str.contains(r'\?', regex=True, na=False).sum() == 0
assert (df_clean['Classe'] == 'NULL').sum() == 0
assert (df_clean['Valor'].astype(str) == 'NULL').sum() == 0
assert df_clean['Valor'].dtype == np.float64
assert len(df_clean) == len(df_raw)
print('Todas as verificações passaram.')

## 4. Carregamento e Validação dos Apontamentos

In [ ]:
df_apon = pd.read_parquet(f'{DATA_DIR}/desenvolver_apontamentos.parquet')
df_apon['Inicio'] = pd.to_datetime(df_apon['Inicio'])
df_apon['Fim'] = pd.to_datetime(df_apon['Fim'])
print(f'Apontamentos: {len(df_apon):,} registros')
print(df_apon['Classe'].value_counts())

In [ ]:
apon_report = validate_apontamentos(df_apon)
print(apon_report.summary)
print(f'Inicio > Fim: {apon_report.inicio_after_fim}')

## 5. Salvando Dados Limpos

In [ ]:
df_clean.to_parquet(f'{DATA_DIR}/cleaned_telemetry.parquet', index=False)
df_apon.to_parquet(f'{DATA_DIR}/cleaned_apontamentos.parquet', index=False)

quality_md = f"""# Quality Report\n\n{report.summary}\n\n{apon_report.summary}"""
with open(f'{DATA_DIR}/quality_report.md', 'w', encoding='utf-8') as f:
    f.write(quality_md)

print('Dados limpos salvos em data/')